In [9]:
import requests
import os
import random

# Step 1: Search for Jungle Babbler recordings using Xeno-Canto API
query = "Jungle Babbler"
url = f"https://xeno-canto.org/api/2/recordings?query={query}"

response = requests.get(url)
data = response.json()

base_path = "C:/Users/maany/Birdie"

In [13]:
file_url = f"https:{rec['file']}"

In [14]:
file_url = f"https://xeno-canto.org{rec['file']}"

In [4]:

# Step 2: Create folders for train and test data
os.makedirs("bird_data/train", exist_ok=True)
os.makedirs("bird_data/test", exist_ok=True)

# Step 3: Select 50 recordings
recordings = data["recordings"][:50]

# Step 4: Randomly select 10 for testing
test_indices = set(random.sample(range(50), 10))


In [ ]:
-------------------------------------------------------------

In [17]:
pip install pydub

  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pydub-0.25.1-py2.py3-none-any.whl (32 kB)



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
    pip install --user librosa

  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached audioread-3.0.1-py3-none-any.whl.metadata (8.4 kB)



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



  Using cached scikit_learn-1.6.1-cp39-cp39-win_amd64.whl.metadata (15 kB)
  Using cached soundfile-0.13.1-py2.py3-none-win_amd64.whl.metadata (16 kB)
  Using cached pooch-1.8.2-py3-none-any.whl.metadata (10 kB)
  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
  Using cached numba-0.60.0-cp39-cp39-win_amd64.whl.metadata (2.8 kB)
  Using cached llvmlite-0.43.0-cp39-cp39-win_amd64.whl.metadata (4.9 kB)
Using cached librosa-0.11.0-py3-none-any.whl (260 kB)
Using cached audioread-3.0.1-py3-none-any.whl (23 kB)
Using cached lazy_loader-0.4-py3-none-any.whl (12 kB)
Using cached numba-0.60.0-cp39-cp39-win_amd64.whl (2.7 MB)
Using cached llvmlite-0.43.0-cp39-cp39-win_amd64.whl (28.1 MB)
Using cached pooch-1.8.2-py3-none-any.whl (64 kB)
Using cached scikit_learn-1.6.1-cp39-cp39-win_amd64.whl (11.2 MB)
Using cached soundfile-0.13.1-py2.py

In [5]:
from pydub import AudioSegment
import os

mp3_folder = "C:/Users/maany/birdie/bird_data/train"
wav_folder = "C:/Users/maany/birdie/data_wav"
os.makedirs(wav_folder, exist_ok=True)

for file in os.listdir(mp3_folder):
    if file.endswith(".mp3"):
        sound = AudioSegment.from_mp3(os.path.join(mp3_folder, file))
        wav_path = os.path.join(wav_folder, file.replace(".mp3", ".wav"))
        sound.export(wav_path, format="wav")

In [6]:
import librosa
import numpy as np

def extract_mel_spectrogram(file_path):
    y, sr = librosa.load(file_path, sr=None)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    return S_db

ModuleNotFoundError: No module named 'librosa'

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_similarity_matrix(files):
    specs = [extract_mel_spectrogram(f).flatten() for f in files]
    specs = np.array(specs)
    sim_matrix = cosine_similarity(specs)
    return sim_matrix

In [ ]:
test_file = "C:/pc/pers/projects/birdie/data_wav/test.wav"
train_files = [f for f in os.listdir(wav_folder) if f != "test.wav"]
train_paths = [os.path.join(wav_folder, f) for f in train_files]

test_spec = extract_mel_spectrogram(test_file).flatten().reshape(1, -1)
train_specs = [extract_mel_spectrogram(f).flatten() for f in train_paths]

scores = cosine_similarity(test_spec, train_specs)[0]
top_indices = np.argsort(scores)[-5:][::-1]

for i in top_indices:
    print(f"Match: {train_files[i]} | Similarity Score: {scores[i]:.2f}")